In [1]:
# Librerias

import re
import nltk

import pandas as pd
import numpy as np

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from nltk.corpus import stopwords

In [2]:
# Cargar datos

train = pd.read_csv('data/train.csv')
eval_df = pd.read_csv('data/eval.csv')

data = train.copy()

print("TAMAÑO DEL DATASET")
print(data.shape)
print(data.head())

print("\nDISTRIBUCIÓN DE LAS DECADAS")
print(data['decade'].value_counts())
print("\nNúmero de decadas:", data['decade'].nunique())

TAMAÑO DEL DATASET
(31403, 2)
                                                text  decade
0  \nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...     164
1  gone. Sus amigos , sus clientes, todo \ncuanto...     182
2  Prefosen quemanera,e per qualesfolpechas deuan...     157
3  Caistro  el  M  a  y  o  r  a  i  .]  Del  ape...     163
4  \nlos  que  panden  macho  ;  y \notros  en  l...     166

DISTRIBUCIÓN DE LAS DECADAS
decade
160    848
172    842
155    836
170    833
167    831
178    831
154    830
157    827
163    827
180    825
168    822
175    817
171    816
165    814
151    812
188    809
179    809
182    808
162    808
174    807
164    804
185    803
184    802
173    802
159    802
181    795
183    794
156    792
161    787
187    787
150    786
152    785
177    782
166    779
158    778
153    775
186    773
169    771
176    754
Name: count, dtype: int64

Número de decadas: 39


In [3]:
# Ejemplos de distintas décadas

for decade in [150, 165, 180]:
    ejemplo = data[data['decade'] == decade]['text'].iloc[0]
    print(f"\n=== Década {decade} ===")
    print(ejemplo[:300])
    print("---")


=== Década 150 ===
efiotnl fiiT’e^ pt\»tf)e 4 trCe et lleene.^^ta^ 
41 tT»íi A*e(leee A(ittc(«t>iieii|l>le iié <oii|ette 
*iW íléiW^* temer pw tnner>>ellmpí«íK> 
---

=== Década 165 ===
Efto fé ha viña en el Confejo de Portagal,c\ qual 
pbí M0'fé ; r precedido del de Aragón , nunca ha queri- 
do concurrir éfl 1 asProce fsione s , be lámanos , ni jun- 
---

=== Década 180 ===

(87) 
obligación  que  las  Ordenanzas  imponen  al  Director 
general  de  la  Armada  sobre  el  zelar  que  se  mejoren 
las  cartas  y  derroteros  en  conformidad  de  las  noticias 
que  deben  dársele  en  quanto  á  los  descubrimientos  de 
nuevas  tierras,  islas,  baxos  y  sondas,  d  r
---


In [5]:
# Limpieza de los datos

# Stopwords en español - nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    # 1. Normalizar saltos de línea y espacios múltiples
    texto = re.sub(r'\n+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    
    # 2. Quitar caracteres que claramente son ruido OCR
    # (símbolos que no son letras, números ni puntuación básica)
    texto = re.sub(r'[^\w\s.,;:!?áéíóúüñÁÉÍÓÚÜÑ]', ' ', texto)
    
    # 3. Strip
    texto = texto.strip().lower()

    # 4. Quitar stopwords
    #palabras = texto.split()
    #palabras = [p for p in palabras if p not in stop_words]

    # 5. Quitar puntuación y números que quedaron sueltos
    texto = re.sub(r'\b[\d.,;:!?]+\b', ' ', texto)
    
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]
    
    # 6. Filtrar tokens de 1 o 2 caracteres (ruido OCR)
    palabras = [p for p in palabras if len(p) > 2]

    return ' '.join(palabras)

data['text_clean'] = data['text'].apply(limpiar_texto)
eval_df['text_clean'] = eval_df['text'].apply(limpiar_texto)

# Verifica el resultado
for decade in [150, 165, 180]:
    ejemplo = data[data['decade'] == decade]['text_clean'].iloc[0]
    print(f"\n=== Década {decade} ===")
    print(ejemplo[:300])

[nltk_data] Downloading package stopwords to C:\Users\Juan
[nltk_data]     David\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



=== Década 150 ===
efiotnl fiit trce lleene. leee ittc iieii iié oii ette íléiw temer tnner ellmpí

=== Década 165 ===
efto viña confejo portagal qual pbí precedido aragón nunca queri concurrir éfl asproce fsione lámanos jun

=== Década 180 ===
obligación ordenanzas imponen director general armada zelar mejoren cartas derroteros conformidad noticias deben dársele quanto descubrimientos nuevas tierras, islas, baxos sondas, rectificación acaso hiciese posiciones locadas. cuidado duda alguna bueno pro vechoso: capaz producir armada real utili


In [6]:
# Palabras más comunes

for decade in [150, 165, 180]:
    textos = ' '.join(data[data['decade'] == decade]['text_clean'])
    palabras = textos.split()
    mas_comunes = Counter(palabras).most_common(15)
    print(f"\n=== Década {decade} ===")
    print(mas_comunes)


=== Década 150 ===
[('carta', 218), ('rey', 145), ('doña', 129), ('juan', 111), ('carlos', 89), ('don', 79), ('felipe', 74), ('conde', 70), ('luis', 65), ('señor', 63), ('diego', 61), ('duque', 60), ('pedro', 58), ('orden', 57), ('francisco', 57)]

=== Década 165 ===
[('mas', 216), ('fus', 125), ('tan', 122), ('don', 111), ('vna', 83), ('fer', 80), ('rey', 79), ('pues', 78), ('dos', 74), ('dicho', 66), ('fino', 63), ('fin', 63), ('dios', 63), ('gran', 55), ('qual', 52)]

=== Década 180 ===
[('mas', 226), ('dos', 109), ('pues', 81), ('bien', 79), ('ser', 66), ('tan', 66), ('don', 66), ('año', 65), ('rey', 63), ('quando', 56), ('per', 56), ('parte', 56), ('mismo', 54), ('tiempo', 51), ('dios', 51)]


In [7]:
# Representación TF-IDF

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    lowercase=False,
    min_df=5,
    sublinear_tf=True
)

X_train = tfidf.fit_transform(data['text_clean'])
y_train = data['decade']

print("\nTAMAÑO DEL DATASET")
print(X_train.shape)


TAMAÑO DEL DATASET
(31403, 359625)


In [ ]:
# Ensamblar modelos (Regresión Logistica y Naive Bayes)

# Entrenar ambos modelos
lr = LogisticRegression(max_iter=1000, C=5.0, solver='saga')
nb = MultinomialNB(alpha=0.1)

# Cross-validation manual para combinar probabilidades
skf = StratifiedKFold(n_splits=5)
accuracies = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    lr.fit(X_tr, y_tr)
    nb.fit(X_tr, y_tr)
    
    # Promediar probabilidades
    proba_ensemble = (0.4 * lr.predict_proba(X_val) + 0.6 * nb.predict_proba(X_val))
    y_pred = lr.classes_[np.argmax(proba_ensemble, axis=1)]
    
    accuracies.append((y_pred == y_val).mean())

print(f"Accuracy ensemble: {np.mean(accuracies):.4f}")

In [ ]:
# Entrenar con todos los datos
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)

# Predecir sobre eval
X_eval = tfidf.transform(eval_df['text_clean'])

proba_ensemble = (0.4 * lr.predict_proba(X_eval) + 0.6 * nb.predict_proba(X_eval))
y_pred_eval = lr.classes_[np.argmax(proba_ensemble, axis=1)]

submission = pd.DataFrame({
    'id': eval_df['id'],
    'answer': y_pred_eval
})

submission.to_csv('submissions/submission_ensemble.csv', index=False)
print(submission.head(10))
print(f"Shape: {submission.shape}")